# Milestone 1: Basic Agent & Triage

This notebook walks through building and testing the core triage node and basic ReAct loop.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify env vars are loaded
print('GOOGLE_API_KEY set:', bool(os.getenv('GOOGLE_API_KEY')))
print('LANGCHAIN_API_KEY set:', bool(os.getenv('LANGCHAIN_API_KEY')))

In [ ]:
# Test the triage node in isolation
from nodes import load_memory, triage
from state import AgentState

test_email = {
    'id': 'nb_001',
    'sender': 'boss@company.com',
    'subject': 'Can you send the report?',
    'body': 'Hi, please send me the Q3 report by end of day. Thanks.',
    'timestamp': '2024-01-15T10:00:00'
}

state: AgentState = {
    'email_input': test_email,
    'messages': [],
    'triage_result': None,
    'draft_response': None,
    'hitl_decision': None,
    'human_edit': None,
    'memory_context': None,
    'final_response': None,
}

# Step 1: load memory
state.update(load_memory(state))
print('Memory:', state['memory_context'])

# Step 2: triage
state.update(triage(state))
print('Triage result:', state['triage_result'])

In [ ]:
# Test the full graph end-to-end (non-interactive — no HITL stdin)
import uuid
from graph import graph

spam_email = {
    'id': str(uuid.uuid4()),
    'sender': 'noreply@spam.com',
    'subject': 'Congratulations! You have won!',
    'body': 'Click here to claim your prize.',
    'timestamp': '2024-01-15T10:05:00'
}

initial_state = {
    'email_input': spam_email,
    'messages': [],
    'triage_result': None,
    'draft_response': None,
    'hitl_decision': None,
    'human_edit': None,
    'memory_context': None,
    'final_response': None,
}

config = {'configurable': {'thread_id': spam_email['id']}}
result = graph.invoke(initial_state, config=config)
print('Final triage:', result['triage_result'])
print('Final response:', result['final_response'])

In [ ]:
# Draw the graph
from IPython.display import Image
try:
    img_data = graph.get_graph().draw_mermaid_png()
    display(Image(img_data))
except Exception as e:
    print('Graph image requires graphviz:', e)
    print(graph.get_graph().draw_mermaid())

In [ ]:
# Run triage accuracy evaluation
from evaluation import run_triage_evaluation
results = run_triage_evaluation()